# BigAlpha 2026 - v13 local-training INFERENCE

**Local-training channel** (wiki "模型本地化训练指南"). This notebook only **loads** the
locally-trained `transformer_model.json` and scores the cloud test period - it does
NOT train. The public leaderboard loads the JSON directly (no retrain -> no timeout,
unlike v10/v11 which trained inside `main` and failed). The private leaderboard
retrains from `transformer_train_local.py`.

Submission directory must contain **all three**:
- this inference notebook (defines `main(datasources, start_date, end_date)`)
- `transformer_train_local.py` (training script, for private retrain)
- `transformer_model.json` (weights + z-score stats + config)

Pipeline is identical to training (`to_canonical(is_local=False)` -> log1p counts
-> per-day /first-close price normalization -> z-score with the **saved** train
mean/std), so there is no train/infer drift. The 25-feature 3-level-book set matches
the local training (cloud 4/5档 are dropped in `to_canonical`). v13 differentiable
soft-rank IC loss was used in training.


In [ ]:
def main(datasources, start_date, end_date):
    """BigAlpha 2026 - v13 local-training INFERENCE.

    Loads the locally-trained transformer_model.json and scores the cloud test
    period. The model is NOT trained here - the public leaderboard loads the
    JSON directly (private leaderboard retrains via transformer_train_local.py).

    Feature pipeline is identical to training (transformer_train_local.py):
    to_canonical(is_local=False) -> log1p counts -> per-day /first_close price
    normalization -> z-score with the training-set mean/std saved in the JSON.
    """
    import os, json, time, numpy as np, pandas as pd, dai
    import torch, torch.nn as nn
    import structlog, warnings
    warnings.filterwarnings('ignore')

    logger = structlog.get_logger()

    def log(msg, **kw):
        ts = time.strftime('%H:%M:%S')
        print(f'[{ts}] {msg} ' + ' '.join(f'{k}={v}' for k, v in kw.items()), flush=True)
        logger.info(msg, **kw)

    # ==================================================================
    # Load locally-trained model
    # ==================================================================
    MODEL_PATH = 'transformer_model.json'
    with open(MODEL_PATH, 'r', encoding='utf-8') as f:
        ckpt = json.load(f)
    FEATURE_COLS = ckpt['feature_cols']
    PRICE_COLS = ckpt['price_cols']
    VOL_COLS = ckpt['vol_cols']
    SEQ_LEN = ckpt['seq_len']
    N_FEAT = len(FEATURE_COLS)
    OHLC_COLS = ['open', 'high', 'low', 'close']
    mean = np.array(ckpt['mean'], dtype=np.float32)
    std = np.array(ckpt['std'], dtype=np.float32)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class StockTransformer(nn.Module):
        def __init__(self, n_feat, d_model, nhead, dim_ff, nlayers, dropout, seq_len):
            super().__init__()
            self.proj = nn.Linear(n_feat, d_model)
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
            enc = nn.TransformerEncoderLayer(
                d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout,
                batch_first=True, activation='gelu')
            self.encoder = nn.TransformerEncoder(enc, num_layers=nlayers)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model // 2), nn.GELU(),
                nn.Linear(d_model // 2, 1))

        def forward(self, x):
            h = self.proj(x) + self.pos[:, :x.size(1), :]
            return self.head(self.encoder(h).mean(dim=1)).squeeze(-1)

    model = StockTransformer(**ckpt['model_cfg']).to(device)
    sd = {k: torch.tensor(np.array(v['data'], dtype=v['dtype']).reshape(v['shape']),
                          device=device)
          for k, v in ckpt['state_dict'].items()}
    model.load_state_dict(sd)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    assert 1e5 <= n_params <= 1e8, f'params {n_params} out of bounds [1e5, 1e8]'
    log('loaded model', path=MODEL_PATH, params=n_params, n_feat=N_FEAT,
        seq_len=SEQ_LEN, device=str(device))

    # ==================================================================
    # Canonical alignment (cloud -> same form training used)
    # ==================================================================
    def to_canonical(df):
        df = df.copy()
        # cloud is already in 元 with NaN missing; drop 4/5档 to match local 3档
        drop = [c for c in df.columns
                if any(c.startswith(p) and c[-1] in '45'
                       for p in ('ask_price', 'bid_price', 'ask_volume',
                                 'bid_volume', 'ask_num_orders', 'bid_num_orders'))]
        df = df.drop(columns=drop, errors='ignore')
        df['key'] = df['instrument']
        return df

    def preprocess(df):
        df = df.copy()
        for c in FEATURE_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
        for c in VOL_COLS:
            df[c] = np.log1p(np.clip(df[c].to_numpy(), 0, None))
        return df

    # ==================================================================
    # Query cloud data
    # ==================================================================
    table = datasources.get('bar5m') or datasources.get('bar1m') \
        or next(iter(datasources.values()))
    INFER_IS_1M = ('bar1m' in str(table).lower()) or ('1m' in str(table).lower())
    buf_s = (pd.to_datetime(start_date) - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
    sql = (f"SELECT date, instrument, {', '.join(FEATURE_COLS)} "
           f"FROM {table} ORDER BY instrument, date")
    raw = dai.query(sql, filters={'date': [buf_s, str(end_date)]}).df()
    log('queried cloud', rows=len(raw), table=str(table), infer_is_1m=INFER_IS_1M)
    if len(raw) == 0:
        raise RuntimeError('empty inference query')

    if INFER_IS_1M:
        raw = raw.copy()
        raw['dt'] = pd.to_datetime(raw['date'])
        raw['bucket'] = raw['dt'].dt.floor('5min')
        agg = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
               'volume': 'sum', 'amount': 'sum'}
        for c in FEATURE_COLS:
            if c not in agg:
                agg[c] = 'last'
        raw = raw.groupby(['instrument', 'bucket'], sort=False).agg(agg).reset_index()
        raw = raw.rename(columns={'bucket': 'date'})
        raw['date'] = pd.to_datetime(raw['date'])

    canon = preprocess(to_canonical(raw))
    for c in OHLC_COLS:
        canon[c] = canon.groupby('key')[c].ffill()
    canon['day'] = canon['date'].dt.normalize()
    sd_ts, ed_ts = pd.to_datetime(start_date), pd.to_datetime(end_date)

    # ==================================================================
    # Build windows (mirror training: per (key,day) last SEQ_LEN bars, /first_close)
    # ==================================================================
    all_X, all_meta = [], []
    for ins, sub in canon.groupby('key', sort=False):
        sub = sub.sort_values('date')
        for day_val, grp in sub.groupby('day', sort=True):
            dt = pd.Timestamp(day_val)
            if dt < sd_ts or dt > ed_ts:
                continue
            if len(grp) < SEQ_LEN:
                continue
            feats = grp[FEATURE_COLS].to_numpy(np.float32)[-SEQ_LEN:]
            ref = float(grp['close'].iloc[0])
            if ref > 0 and np.isfinite(ref):
                for ci_f, c in enumerate(FEATURE_COLS):
                    if c in PRICE_COLS:
                        feats[:, ci_f] = feats[:, ci_f] / ref
            all_X.append(feats)
            all_meta.append((dt, ins))
    if not all_X:
        raise RuntimeError('no inference windows built')
    X = np.stack(all_X).astype(np.float32)
    X = ((X - mean) / std).astype(np.float32)
    meta_df = pd.DataFrame(all_meta, columns=['date', 'instrument'])
    log('windows built', samples=len(X), days=meta_df['date'].nunique(),
        instruments=meta_df['instrument'].nunique())
    del all_X, canon, raw

    # ==================================================================
    # Predict + per-date cross-section z-score
    # ==================================================================
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), 1024):
            xb = torch.from_numpy(np.ascontiguousarray(X[i:i+1024])).to(device)
            preds.append(model(xb).cpu().numpy())
    meta_df['score'] = np.concatenate(preds).astype(np.float64)
    meta_df['score'] = meta_df.groupby('date')['score'].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-8))
    # clip per-date z-scores: the soft-rank loss doesn't constrain prediction
    # magnitude, so a few outlier-feature instruments get extreme raw preds ->
    # extreme z-scores. Monotonic clip preserves ranks (IC) but caps the tails.
    meta_df['score'] = meta_df['score'].clip(-5.0, 5.0)

    # ==================================================================
    # Align to the official stock universe
    # ==================================================================
    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]}).df()
    result = (pd.merge(meta_df, stk, on=['date', 'instrument'], how='inner')
                .replace([np.inf, -np.inf], np.nan)
                .dropna(subset=['score'])
                .drop_duplicates(['date', 'instrument'])
                [['date', 'instrument', 'score']]
                .reset_index(drop=True))

    log('=' * 60)
    log('DONE', rows=len(result), days=result['date'].nunique(),
        instruments=result['instrument'].nunique(),
        score_mean=round(float(result['score'].mean()), 6),
        score_std=round(float(result['score'].std()), 6),
        score_min=round(float(result['score'].min()), 6),
        score_max=round(float(result['score'].max()), 6))
    log('=' * 60)
    return result


if __name__ == '__main__':
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    sd, ed = '2024-01-01', '2024-03-31 23:59:59'
    out = main(datasources, sd, ed)
    print(out.head(10))
    print(f'rows={len(out)} days={out.date.nunique()} ins={out.instrument.nunique()}')
    print(out.score.describe())
